# Feature Audit – Kiểm tra `household_featured.xlsx`

Notebook này kiểm tra lại các điểm reviewer cảnh báo:

1. `NHAN_KHAU` có giá trị bất thường, ví dụ max = 505.
2. Các biến số có tương quan tuyến tính cao.
3. Một số feature gần như mang cùng thông tin, cần loại bớt trước K-Means và XGBoost.

**Input:** `household_featured.xlsx`  
**Output:** `feature_audit.xlsx`


## 0. Khởi tạo thư viện, đường dẫn và đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.stats.outliers_influence import variance_inflation_factor

INPUT_FILE = r"C:\NguyenHoangThanhTam\data\model\household_featured.xlsx"

OUTPUT_DIR = Path(r"C:\NguyenHoangThanhTam\data\feature_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "feature_audit.xlsx"

df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())


Shape: (26953, 34)


,HOUSEHOLD_ID,STT_HO,HUYEN,XA,PHAN_LOAI_HO,NHAN_KHAU,NHAN_KHAU_GOC_MAX,SO_CHU_HO,SO_XA_TRONG_HO,SO_HUYEN_TRONG_HO,...,TY_LE_PHU_THUOC,HO_TEN_CHU_HO,TUOI_CHU_HO,GIOI_TINH_CHU_HO,DAN_TOC_CHU_HO,NHOM_TUOI_CHU_HO,TUOI_TRUNG_BINH_HO,TUOI_LON_NHAT,TUOI_NHO_NHAT,DO_LECH_TUOI
0,1,1,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,1,1.0,1,1,1,...,NaN,TRƯƠNG THỊ LÙN,87.0,Nữ,Kinh,65+,87.000000,87.0,87.0,0.0
1,2,2,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,4,4.0,1,1,1,...,1.0,HOÀNG VĂN TRƯỜNG,46.0,Nam,Kinh,35-49,29.750000,46.0,13.0,33.0
2,3,3,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,2,2.0,1,1,1,...,NaN,TRẦN VĂN LONG,66.0,Nam,Kinh,65+,60.500000,66.0,55.0,11.0
3,4,4,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,1,1.0,1,1,1,...,NaN,NGUYỄN THỊ TƯƠI,80.0,Nữ,Kinh,65+,80.000000,80.0,80.0,0.0
4,5,5,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,6,6.0,1,1,1,...,5.0,ĐỖ VĂN PHÚC,66.0,Nam,Kinh,65+,33.333333,66.0,10.0,56.0


['HOUSEHOLD_ID', 'STT_HO', 'HUYEN', 'XA', 'PHAN_LOAI_HO', 'NHAN_KHAU', 'NHAN_KHAU_GOC_MAX', 'SO_CHU_HO', 'SO_XA_TRONG_HO', 'SO_HUYEN_TRONG_HO', 'CO_DONG_CHU_HO', 'SOURCE_ROW_MIN', 'SOURCE_ROW_MAX', 'SO_NAM', 'SO_NU', 'SO_MAM_NON', 'SO_HOC_DUONG', 'SO_LAO_DONG', 'SO_U50', 'SO_PHU_THUOC', 'TY_LE_MAM_NON', 'TY_LE_HOC_DUONG', 'TY_LE_LAO_DONG', 'TY_LE_U50', 'TY_LE_PHU_THUOC', 'HO_TEN_CHU_HO', 'TUOI_CHU_HO', 'GIOI_TINH_CHU_HO', 'DAN_TOC_CHU_HO', 'NHOM_TUOI_CHU_HO', 'TUOI_TRUNG_BINH_HO', 'TUOI_LON_NHAT', 'TUOI_NHO_NHAT', 'DO_LECH_TUOI']


## 1. Kiểm tra `NHAN_KHAU` bất thường

Mục tiêu: xác định các hộ có quy mô nhân khẩu quá lớn, đặc biệt trường hợp reviewer ghi nhận `NHAN_KHAU max = 505`.

Nếu `NHAN_KHAU` quá lớn bất thường, cần kiểm tra lại:
- `STT_HO` có bị gộp sai không?
- Có phải một mã hộ bị dùng cho nhiều hộ khác nhau không?
- Có phải lỗi điền/ffill `STT_HO` hoặc `NHAN_KHAU` không?


#### 1.1. `NHAN_KHAU` có giá trị bất thường không? (Bảng thống kê)

In [2]:
# CELL 1 - Tính toán, không vẽ

nk_desc = df["NHAN_KHAU"].describe().to_frame("NHAN_KHAU")

top_nk = (
    df.sort_values("NHAN_KHAU", ascending=False)
      [["STT_HO", "HUYEN", "XA", "PHAN_LOAI_HO", "NHAN_KHAU",
        "SO_NAM", "SO_NU", "SO_LAO_DONG", "SO_U50", "SO_PHU_THUOC"]]
      .head(30)
)

print("Thống kê NHAN_KHAU:")
display(nk_desc)

print("Top hộ có NHAN_KHAU lớn nhất:")
display(top_nk)

print(f"NHAN_KHAU max = {df['NHAN_KHAU'].max():,.0f}")


Thống kê NHAN_KHAU:


,NHAN_KHAU
count,26953.000000
mean,3.653693
std,2.183086
min,1.000000
25%,2.000000
50%,4.000000
75%,5.000000
max,21.000000


Top hộ có NHAN_KHAU lớn nhất:


,STT_HO,HUYEN,XA,PHAN_LOAI_HO,NHAN_KHAU,SO_NAM,SO_NU,SO_LAO_DONG,SO_U50,SO_PHU_THUOC
25933,21101,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,21,9,12,8,1,13
18112,16078,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,18,12,6,8,1,10
4469,4280,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,17,11,6,6,2,10
25947,21105,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,17,9,8,9,1,8
4199,4025,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,16,7,9,5,0,11
25963,21107,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,16,8,8,9,2,7
17804,15777,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,8,7,7,2,8
439,436,Huyện Cư Jút,ĐẮK D'RÔNG,Hộ nghèo,15,3,12,6,3,9
6792,6500,Huyện Đắk Glong,ĐẮK SOM,Hộ nghèo,15,7,8,7,0,7
18138,16104,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,11,4,8,2,7


NHAN_KHAU max = 21


#### 1.2. Xác định outlier `NHAN_KHAU` bằng IQR

In [3]:
# CELL 1 - Tính toán

Q1 = df["NHAN_KHAU"].quantile(0.25)
Q3 = df["NHAN_KHAU"].quantile(0.75)
IQR = Q3 - Q1
upper_iqr = Q3 + 1.5 * IQR

outlier_nk = (
    df[df["NHAN_KHAU"] > upper_iqr]
    .sort_values("NHAN_KHAU", ascending=False)
    [["STT_HO", "HUYEN", "XA", "PHAN_LOAI_HO", "NHAN_KHAU",
      "SO_NAM", "SO_NU", "SO_LAO_DONG", "SO_U50", "SO_PHU_THUOC"]]
)

outlier_summary = pd.DataFrame({
    "Chi_tieu": [
        "Q1",
        "Q3",
        "IQR",
        "Nguong_outlier_IQR",
        "So_ho_outlier",
        "NHAN_KHAU_max"
    ],
    "Gia_tri": [
        Q1,
        Q3,
        IQR,
        upper_iqr,
        len(outlier_nk),
        df["NHAN_KHAU"].max()
    ]
})

display(outlier_summary)
display(outlier_nk.head(50))


,Chi_tieu,Gia_tri
0,Q1,2.0
1,Q3,5.0
2,IQR,3.0
3,Nguong_outlier_IQR,9.5
4,So_ho_outlier,376.0
5,NHAN_KHAU_max,21.0


,STT_HO,HUYEN,XA,PHAN_LOAI_HO,NHAN_KHAU,SO_NAM,SO_NU,SO_LAO_DONG,SO_U50,SO_PHU_THUOC
25933,21101,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,21,9,12,8,1,13
18112,16078,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,18,12,6,8,1,10
25947,21105,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,17,9,8,9,1,8
4469,4280,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,17,11,6,6,2,10
4199,4025,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,16,7,9,5,0,11
25963,21107,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,16,8,8,9,2,7
25636,21001,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,5,10,9,0,6
25967,21108,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,7,8,9,1,6
6980,6676,Huyện Đắk Glong,ĐẮK SOM,Hộ nghèo,15,8,7,7,2,8
6718,6426,Huyện Đắk Glong,ĐẮK SOM,Hộ nghèo,15,9,6,6,0,9


## 2. Kiểm tra tương quan giữa các feature số

Mục tiêu: phát hiện các cặp feature gần như tuyến tính với nhau, ví dụ:
- `TY_LE_LAO_DONG` và `TY_LE_PHU_THUOC`
- `SO_PHU_THUOC` và tổng các nhóm phụ thuộc
- `SO_NAM`, `SO_NU` và `NHAN_KHAU`


In [4]:
numeric_cols = [
    "NHAN_KHAU",
    "SO_NAM",
    "SO_NU",
    "SO_MAM_NON",
    "SO_HOC_DUONG",
    "SO_LAO_DONG",
    "SO_U50",
    "SO_PHU_THUOC",
    "TY_LE_MAM_NON",
    "TY_LE_HOC_DUONG",
    "TY_LE_LAO_DONG",
    "TY_LE_U50",
    "TY_LE_PHU_THUOC",
    "TUOI_CHU_HO",
    "TUOI_TRUNG_BINH_HO",
    "TUOI_LON_NHAT",
    "TUOI_NHO_NHAT",
    "DO_LECH_TUOI"
]

numeric_cols = [c for c in numeric_cols if c in df.columns]

print("Numeric features:", numeric_cols)


Numeric features: ['NHAN_KHAU', 'SO_NAM', 'SO_NU', 'SO_MAM_NON', 'SO_HOC_DUONG', 'SO_LAO_DONG', 'SO_U50', 'SO_PHU_THUOC', 'TY_LE_MAM_NON', 'TY_LE_HOC_DUONG', 'TY_LE_LAO_DONG', 'TY_LE_U50', 'TY_LE_PHU_THUOC', 'TUOI_CHU_HO', 'TUOI_TRUNG_BINH_HO', 'TUOI_LON_NHAT', 'TUOI_NHO_NHAT', 'DO_LECH_TUOI']


#### 2.1. Ma trận tương quan Pearson giữa các biến số (Heatmap)

In [5]:
# CELL 1 - Tính toán

corr = df[numeric_cols].corr()

display(corr.round(3))


,NHAN_KHAU,SO_NAM,SO_NU,SO_MAM_NON,SO_HOC_DUONG,SO_LAO_DONG,SO_U50,SO_PHU_THUOC,TY_LE_MAM_NON,TY_LE_HOC_DUONG,TY_LE_LAO_DONG,TY_LE_U50,TY_LE_PHU_THUOC,TUOI_CHU_HO,TUOI_TRUNG_BINH_HO,TUOI_LON_NHAT,TUOI_NHO_NHAT,DO_LECH_TUOI
NHAN_KHAU,1.000,0.767,0.782,0.218,0.671,0.788,0.228,0.778,-0.000,0.133,0.084,-0.201,0.351,-0.122,-0.227,0.269,-0.502,0.721
SO_NAM,0.767,1.000,0.321,0.156,0.502,0.618,0.174,0.582,-0.003,0.100,0.085,-0.173,0.237,-0.113,-0.190,0.180,-0.399,0.540
SO_NU,0.782,0.321,1.000,0.186,0.550,0.597,0.165,0.628,0.006,0.114,0.040,-0.143,0.288,-0.077,-0.170,0.224,-0.377,0.563
SO_MAM_NON,0.218,0.156,0.186,1.000,0.108,0.074,-0.046,0.277,0.739,-0.017,-0.096,-0.111,0.188,-0.119,-0.205,-0.087,-0.234,0.128
SO_HOC_DUONG,0.671,0.502,0.550,0.108,1.000,0.269,-0.197,0.818,-0.027,0.682,-0.236,-0.379,0.597,-0.371,-0.502,-0.157,-0.569,0.364
SO_LAO_DONG,0.788,0.618,0.597,0.074,0.269,1.000,0.018,0.269,-0.071,-0.162,0.556,-0.331,-0.282,-0.233,-0.190,0.177,-0.400,0.538
SO_U50,0.228,0.174,0.165,-0.046,-0.197,0.018,1.000,0.365,-0.088,-0.372,-0.271,0.735,0.334,0.700,0.597,0.742,0.309,0.445
SO_PHU_THUOC,0.778,0.582,0.628,0.277,0.818,0.269,0.365,1.000,0.079,0.408,-0.384,0.041,0.739,0.043,-0.166,0.254,-0.395,0.607
TY_LE_MAM_NON,-0.000,-0.003,0.006,0.739,-0.027,-0.071,-0.088,0.079,1.000,-0.065,-0.142,-0.102,0.159,-0.144,-0.219,-0.194,-0.181,-0.025
TY_LE_HOC_DUONG,0.133,0.100,0.114,-0.017,0.682,-0.162,-0.372,0.408,-0.065,1.000,-0.466,-0.438,0.609,-0.488,-0.659,-0.528,-0.558,-0.009


#### 2.2. Liệt kê các cặp feature có tương quan cao

In [6]:
# CELL 1 - Tính toán

threshold = 0.80

pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) >= threshold:
            pairs.append({
                "Feature_1": corr.columns[i],
                "Feature_2": corr.columns[j],
                "Correlation": round(val, 3),
                "Abs_Correlation": round(abs(val), 3)
            })

corr_pairs = (
    pd.DataFrame(pairs)
    .sort_values("Abs_Correlation", ascending=False)
    .reset_index(drop=True)
)

display(corr_pairs)


,Feature_1,Feature_2,Correlation,Abs_Correlation
0,TY_LE_LAO_DONG,TY_LE_PHU_THUOC,-0.874,0.874
1,TUOI_TRUNG_BINH_HO,TUOI_NHO_NHAT,0.873,0.873
2,TY_LE_U50,TUOI_TRUNG_BINH_HO,0.862,0.862
3,TUOI_CHU_HO,TUOI_LON_NHAT,0.849,0.849
4,SO_HOC_DUONG,SO_PHU_THUOC,0.818,0.818


## 3. Kiểm tra đa cộng tuyến bằng VIF

VIF giúp phát hiện feature nào có thể được giải thích tốt bởi các feature còn lại.

Quy ước tham khảo:
- VIF < 5: ổn
- 5 ≤ VIF < 10: cần chú ý
- VIF ≥ 10: đa cộng tuyến mạnh


In [7]:
# CELL 1 - Tính toán

X_vif = df[numeric_cols].copy()

X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
X_vif = X_vif.fillna(X_vif.median(numeric_only=True))

constant_cols = [c for c in X_vif.columns if X_vif[c].nunique(dropna=False) <= 1]
X_vif = X_vif.drop(columns=constant_cols, errors="ignore")

vif_df = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_df = vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)

display(vif_df)


c:\python\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,VIF
0,SO_HOC_DUONG,inf
1,SO_MAM_NON,inf
2,SO_PHU_THUOC,inf
3,SO_U50,inf
4,TUOI_LON_NHAT,3.933851e+04
5,DO_LECH_TUOI,1.688012e+04
6,TUOI_NHO_NHAT,1.126343e+04
7,NHAN_KHAU,2.692261e+02
8,TUOI_TRUNG_BINH_HO,2.061004e+02
9,SO_LAO_DONG,7.977234e+01


## 3. Xuất `feature_audit.xlsx`



In [10]:
# CELL 1 - Xuất output

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    nk_desc.reset_index().rename(columns={"index": "Statistic"}).to_excel(
        writer,
        sheet_name="NHAN_KHAU_Describe",
        index=False
    )

    top_nk.to_excel(
        writer,
        sheet_name="Top_NHAN_KHAU",
        index=False
    )

    outlier_summary.to_excel(
        writer,
        sheet_name="Outlier_Summary",
        index=False
    )

    outlier_nk.to_excel(
        writer,
        sheet_name="Outlier_NHAN_KHAU",
        index=False
    )

    corr.round(4).to_excel(
        writer,
        sheet_name="Correlation_Matrix"
    )

    corr_pairs.to_excel(
        writer,
        sheet_name="High_Correlation",
        index=False
    )

    vif_df.to_excel(
        writer,
        sheet_name="VIF",
        index=False
    )

print("Đã xuất feature_audit.xlsx")
print(f"Vị trí: {OUTPUT_FILE}")


Đã xuất feature_audit.xlsx
Vị trí: C:\NguyenHoangThanhTam\data\feature_audit\feature_audit.xlsx
